### This review covers `Training language models to follow instructions with human feedback` (in particular) and `Reinforcement Learning with Human Feedback` in LLM (in general)
https://proceedings.neurips.cc/paper_files/paper/2022/file/b1efde53be364a73914f58805a001731-Paper-Conference.pdf

Large part of the review will provide explanations to the following:

![image.png](static/rlhf-review/the-gist.png)

> ## Reinforcement Learning recap

![image.png](static/rlhf-review/supermario.png)

**Reinforcement Learning (RL)** studies how an *agent* should act in an *environment* to maximize cumulative reward.

- **Model (environment dynamics + rewards).**

  A mathematical description of how the world evolves and pays the agent. Often formalized as a Markov Decision Process (MDP)
  $$
  \mathcal{M} = (\mathcal{S}, \mathcal{A}, P, r, \gamma),
  $$
  where:
  - $\mathcal{S}$ = states, $\mathcal{A}$ = actions,
  - $P(s' \mid s,a)$ = transition dynamics,
  - $r(s,a)$ (or $r(s,a,s')$) = immediate reward,
  - $\gamma \in [0,1]$ = discount factor.

- **Policy.**

  A decision rule mapping states to actions. Stochastic policy:
  $$
  \pi(a\mid s) = \Pr\{A_t=a \mid S_t=s\}.
  $$
  Deterministic policy: $a = \mu(s)$.

- **Return.**

  Discounted cumulative reward from time $t$:
  $$
  G_t = \sum_{k=0}^{H} \gamma^k\, R_{t+k+1}.
  $$
  where:
  - $k$ is the number of steps into the future from $t$
  - $H$ is the time horizon (e.g., the number of steps until the episode ends).

- **Value Function (state value).**

  Expected future return *from a state* when following $\pi$:
  $$
  V^\pi(s) = \mathbb{E}_\pi \big[ G_t \mid S_t = s \big].
  $$

- **Action-Value Function (Q-value).**

  Expected future return from *taking action $a$ in state $s$ and then following $\pi$*:
  $$
  Q^\pi(s,a) = \mathbb{E}_\pi \big[ G_t \mid S_t = s, A_t = a \big].
  $$

- **Advantage Function.**

  How much better an action is than average at a state:
  $$
  A^\pi(s,a) = Q^\pi(s,a) - V^\pi(s).
  $$

- **Bellman Equations.**

  Recursive definitions linking values across time:
  $$
  V^\pi(s)= \sum_a \pi(a\mid s)\sum_{s'} P(s'\mid s,a)\big[r(s,a,s')+\gamma V^\pi(s')\big],
  $$
  $$
  Q^\pi(s,a)= \sum_{s'} P(s'\mid s,a)\big[r(s,a,s')+\gamma \sum_{a'} \pi(a'\mid s') Q^\pi(s',a')\big].
  $$

- **Objective.**

  Maximize expected return under $\pi$; e.g., start-state value $J(\pi)=\mathbb{E}_{S_0}[V^\pi(S_0)]$ or average episodic return.

- **Learning Paradigms (brief).**
  - *Model-free* vs *model-based* (learn/plan with or without $P,r$).
  - *Value-based* (learn $V,Q$), *policy-based* (optimize $\pi$ directly), or *actor-critic* (both).
  - *On-policy* (learn from data generated by the current $\pi$) vs *off-policy* (learn about a target policy from other behavior).

> **In words:** RL formalizes sequential decision making with a model of transitions and rewards, a policy that maps states to actions, and value functions that quantify expected future reward when following a policy.


> ## Reinforcement Learning in LLM

### 1. Reminder: Advantage in Standard RL

In classical reinforcement learning:

$$
A_t = Q(s_t, a_t) - V(s_t)
$$

* $Q(s_t, a_t)$: expected return if we take action $a_t$ at state $s_t$.
* $V(s_t)$: expected return from state $s_t$, averaged over possible actions.

Thus, $A_t$ measures whether this particular action is better or worse than expected.

---

### 2. The LLM Setting: Reward is Sequence-Level

In RLHF for LLMs:

* The **reward model (RM)** produces a single scalar $R(x, y)$ after the model generates a completion $y$ to a prompt $x$.
* This reward is **not available at the token level**.

Therefore, $Q$ and $V$ cannot be directly observed per step.

---

### 3. How PPO (Proximal Policy Optimization) Algorithm Handles This in LLMs

The process is as follows:

1. **Generate a full sequence**: from prompt $x = (p_1, p_2, \dots, p_N)$, the LLM produces tokens $y = a_1, a_2, \dots, a_T$.

2. **Compute sequence reward**: feed $(x, y)$ into the reward model, yielding a scalar $R$.

3. **Compute returns per timestep**: since every token contributed to the sequence, propagate $R$ back to each timestep $t$:

$$
\hat{R}_t = R \quad \forall t
$$

4. **Value function baseline**: the critic network predicts $V(s_t)$, the expected reward given the current context.

5. **Advantage estimates**: form

$$
A_t = \hat{R}_t - V(s_t).
$$

To reduce variance and improve credit assignment, **Generalized Advantage Estimation (GAE)** is used:

$$
A_t^{\text{GAE}} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \, \delta_{t+l},
\quad \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)
$$

**Explanation of terms:**

* $A_t^{\text{GAE}}$: the smoothed advantage estimate at time $t$.
* $l$: summation index, representing how far ahead in time we look from $t$.
* $\gamma \in [0,1]$: the discount factor, controlling how much future rewards are weighted relative to immediate ones.
* $\lambda \in [0,1]$: the GAE parameter, interpolating between high-bias/low-variance (small $\lambda$) and low-bias/high-variance (large $\lambda$) estimates.
* $\delta_t$: temporal-difference (TD) residual at time $t$.
* $r_t$: the reward at time step $t$. In LLMs, $r_t = 0$ except at the final token.
* $V(s_t)$: estimated value of state $s_t$ from the critic.
* $V(s_{t+1})$: estimated value of the next state.

---

### 4. When is $A_t$ Computed?

* **Not at inference time**: during deployment, the model just samples tokens; no reward model is consulted.
* **During training episodes**: when sampling rollouts for PPO, $A_t$ is computed **per token** using the RM + critic baseline.

Hence, $A_t$ is computed per token, but only after generating the full sequence and receiving the sequence-level reward.

---

### 5. Intuition

* Early tokens receive advantage values shaped by the critic, since the raw RM only provides a single scalar.
* GAE smooths credit assignment: if the ending is bad, earlier tokens are penalized less if they were reasonable given their context.


> ## What is a Reward Model and Value Function?

## 1. The Reward Model (RM)

**What it is:**

* The RM is a **separate neural network** (often initialized from the same pretrained LM backbone) fine-tuned to predict **human preference scores**.
* It’s trained on *pairwise comparison data* from annotators: given two model outputs for the same prompt, which one is better?
* The RM learns a scalar function

  $$
  R_\phi(x, y) \in \mathbb{R}
  $$

  where $x$ = prompt, $y$ = completion, and $\phi$ are the RM parameters.

**How it’s trained:**

1. Collect human feedback: for prompt $x$, humans rank completions $y^{(1)}, y^{(2)}$.
2. Optimize RM with a Bradley–Terry or logistic loss:

   $$
   \mathcal{L}_{\text{RM}} = - \mathbb{E}\_{(x, y^+, y^-)} \Big[ \log \sigma(R_\phi(x,y^+) - R_\phi(x,y^-)) \Big]
   $$

   where $y^+$ is preferred over $y^-$.
3. After training, the RM acts as a **proxy for human judgment**.

**In real scenarios:**

* RM gives a scalar score to *entire sequences*.
* Example: For the prompt *“Write a polite refusal”*, the RM might assign a higher score to *“I’m sorry, but I can’t share that information”* than *“Nope, can’t tell you.”*

---

## 2. The Value Function (Critic)

**What it is:**

* The value function $V_\theta(s)$ is usually implemented as a **learned head on top of the LLM** (same model used for generation).
* It predicts the **expected reward** (from the RM) given the **current state** (the prompt + generated prefix so far).

**Why it’s needed:**

* In PPO, the critic provides a baseline to compute advantages:

  $$
  A_t = \hat{R}_t - V(s_t)
  $$
* Without it, the variance of updates would be huge, since the reward signal comes only at the end of a sequence.

**How it’s trained:**

* During rollouts, for each generated sequence, the RM gives a scalar reward $R$.
* This reward is **broadcast across timesteps**, and the critic is trained with regression loss:

  $$
  \mathcal{L}_{\text{value}} = \sum_t \big( V(s_t) - \hat{R}_t \big)^2
  $$
* Over time, the critic learns to approximate the RM’s expected score for any partial context.

**In real scenarios:**

* Example: At the middle of generating *“I’m sorry, but I…”*, the critic might already estimate that the continuation will likely get a high RM score, compared to *“No, I…”*.

---

## 3. Relationship Between Them

* **Reward Model** = proxy for human preferences, trained separately, frozen during PPO fine-tuning.
* **Value Function (Critic)** = baseline predictor, trained jointly with PPO to reduce variance and stabilize updates.
* They work together as follows:

  * RM provides the **final scalar reward**.
  * Critic provides **stepwise estimates** to compute per-token advantages.

---

Allegory:

* **RM ≈ “Human judge in a box.”**
* **Value head ≈ “Running estimator” of that judge’s future opinion at each token step.**


> PPO Update

### 1. Inputs to PPO Update
You’re right: the PPO update block consumes

* Scalar reward ($R$) → provided by the reward model.
* Value estimate ($V(s_t)$) → from the value head on the policy LLM.
* Advantage ($A_t$) → computed from ($R$) and ($V(s_t)$) via GAE.

These define the *targets* against which the current policy is updated.

### 2. Probability ratio ($r_t(\theta)$)
The probability ratio is central to PPO’s clipped objective:
$$r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)}.$$

* **Numerator:**
    The *current policy* (LLM after an update step) produces a softmax distribution over tokens.
    The probability of the actually chosen token ($a_t$) under the *new* parameters ($\theta$) is extracted.
* **Denominator:**
    The *old policy* (LLM before the PPO update, frozen copy) also assigns a probability to ($a_t$).
    This is cached during rollout generation.

**Intuition:**
* If ($r_t(\theta) \approx 1$), the new and old models agree.
* If ($r_t(\theta) \gg 1$), the new policy made the token much more likely → update could be too aggressive.
* If ($r_t(\theta) \ll 1$), the new policy suppressed it → update could again destabilize.

Note: PPO clips this ratio to ($[1 - \epsilon, 1 + \epsilon]$), ensuring the new policy doesn’t drift too far.

### 3. Entropy
Entropy is added to the objective as a *regularizer*:
$$S(\pi_\theta) = - \sum_{a} \pi_\theta(a \mid s_t) \log \pi_\theta(a \mid s_t).$$

* **Origin:**
    Computed directly from the policy distribution (the softmax logits at each step).
    It measures how “spread out” the probability mass is.
* **Role:**
    Encourages exploration (avoids the model collapsing into overly deterministic outputs).
    Prevents premature convergence to narrow token distributions.
    In LLM RLHF, it keeps generations diverse instead of reward-hacked.
* **Practical detail:**
    Weighted by a coefficient ($c_2$) in the PPO loss:
    $$
    L(\theta) = L^{\text{CLIP}}(\theta) - c_1 L^{\text{value}} + c_2 S(\pi_\theta).
    $$

### 4. Putting it all together: PPO Update
Inside PPO, at each training iteration:

1.  **Rollouts:** Generate completions ($y_{i}$) to prompts ($x_{i}$) using the old policy ($\pi_{\theta_{\text{old}}}$).
2.  **Rewards:** Get scalar ($R$) from the reward model.
3.  **Value baseline:** Compute ($V(s_t)$).
4.  **Advantages:** Estimate ($A_t$) via GAE.
5.  **Ratios:** For each action (token), compute ($r_t(\theta)$) comparing current vs old policy probabilities.
6.  **Clipped policy loss:**
    $$
    L^{\text{CLIP}}(\theta) = \mathbb{E}_t\Big[\min\big(r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t\big)\Big].
    $$
7.  **Value loss:** Train critic to predict ($R$).
8.  **Entropy bonus:** Add ($c_2 S(\pi_\theta)$).
9.  **Backprop:** Compute gradients
    * Policy gradients ($\nabla_\theta L^{\text{PPO}}_\pi$) → LM head + backbone.
    * Value gradients ($\nabla_\theta L^{\text{value}}$) → Value head + backbone.

In a nutshell:

* **Probability ratio** originates from comparing the new and old policy distributions over the *same token choices*.
* **Entropy** originates directly from the current policy’s softmax distribution over the vocabulary, used as a regularization signal.

> Training step-by-step: using an example of 10 human prompts + 4 ranked output for each prompt

### 1. Reward Model (RM) Training
* For each prompt ($x_i$), annotators provide rankings over 4 candidate outputs ($\{y_i^{(1)}, \dots, y_i^{(4)}\}$).
* The reward model ($R_\phi(x,y)$) (a separate LM) is trained with a pairwise preference loss:
    $$
    \mathcal{L}_{\text{RM}} = -\log \sigma\big(R_\phi(x,y^+) - R_\phi(x,y^-)\big),
    $$
    where ($y^+$) is ranked higher than ($y^-$).
* After training, the RM is **frozen**; it can now assign scalar rewards to new completions.

![image.png](static/rlhf-review/reward-model-training.png)


### 2. Rollout Generation (Policy LM)
* A batch of 10 prompts ($\{x_1, \dots, x_{10}\}$) is fed into the current policy LM ($\pi_\theta$).
* The LM generates completions token-by-token via softmax sampling, producing trajectories ($\tau_i = (s_t, a_t)$).


![image.png](static/rlhf-review/rollout-generation.png)


### 3. Reward Assignment
* Each generated sequence ($(x_i, y_i)$) is scored by the frozen reward model:
    $$
    R_i = R_\phi(x_i, y_i).
    $$
* This scalar is broadcast across timesteps for that sequence.

### 4. Advantage Estimation
* The policy LM’s **value head** predicts ($V_\theta(s_t)$) for each prefix.
* Advantages are computed using GAE:
    $$
    A_t = \text{GAE}(R_i, V_\theta(s_t)).
    $$

### 5. PPO Update
* Compute probability ratios between new and old policies:
    $$
    r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)}.
    $$
* The PPO loss combines:
    * **Clipped policy loss** (actor, from logits head),
    * **Value loss** (critic, from value head),
    * **Entropy bonus** (exploration).

### 6. Backpropagation
* Gradients from the PPO loss are backpropagated:
    * **Policy gradients** update LM logits + shared backbone.
    * **Value gradients** update value head + shared backbone.
* Parameters ($\theta$) are updated; ($\pi_{\theta_{\text{old}}}$) is refreshed for the next iteration.

**Summary:**

* Human rankings train the reward model
* The frozen reward model provides scalar rewards for generated completions
* The PPO loop (policy LM + value head) uses these rewards with GAE to compute advantages, probability ratios, and entropy, and applies backpropagation to update the language model.

![image.png](static/rlhf-review/training-architecture.png)